# All planes + weak labels — fixed and runnableThis is `with-all-planes-weak-label.ipynb` with the blockers removed. Yourarchitecture, your label-merge logic and your Dataset are kept as you wrotethem. Five things were broken or missing; each fix is marked **FIX** in thecell that contains it.| # | problem | fix ||---|---|---|| 1 | feature caching covered 14,010 series (all 4,407 studies x 3 planes) and never finished | cache only the 1,778 studies actually trained or validated on; with `PLANES = ["axial_fs"]` that is 1,778 series instead of 14,010 || 2 | `series_by_study` used by the Dataset but never built | built from the cache paths || 3 | `clean_df` / `weak_df` / `TRAIN_SERIES_ROOT` / `use_amp` never defined; `Dataset`, `DataLoader` never imported | defined and imported || 4 | `prepare_multiplane_inputs` left `lengths` on CPU while `plane_present` went to the GPU, so `encode_plane` raised a device mismatch on the first batch | `lengths` moved to `DEVICE`; your model already calls `vl.cpu()` where `pack_padded_sequence` needs it || 5 | validation was a single 11-study split | 5-fold CV over all 58 labelled studies, out-of-fold predictions pooled |Fix 5 is the one worth arguing about, so: on 11 studies a single studyflipping moves macro AUC by several points, and on the 56-study CV simplyappending a **constant, information-free column** to the features moves it byabout 0.01. Nothing below roughly +0.02 to +0.03 is measurable on this data.An 11-study number cannot tell a real gain from noise, which is why thereference figures below come from the pooled CV.Reference, same protocol, mean-pooled features into a small head:    sagittal only      0.7332 +- 0.0016    axial only         0.7514 +- 0.0105    all three planes   0.7627 +- 0.0036Those use 56 studies (sagittal-restricted); this notebook uses all 58, since`plane_present` already handles a study that lacks a plane. Close but not anidentical denominator.

In [ ]:
import os, json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader          # FIX 3: were missing
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import pydicom
from PIL import Image
from torchvision import models, transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WEAK_CSV = ("/kaggle/input/datasets/addy789/weak-labels-by-study/"
            "weak_labels_by_study.csv")
TRAIN_SERIES_ROOT = os.path.join(DATA_DIR, "train_series")   # FIX 3
FEATURE_CACHE = "/kaggle/working/feature_cache"
os.makedirs(FEATURE_CACHE, exist_ok=True)

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

# Start with axial only: it is the single cheapest run AND the only plane
# present for all 4,407 studies (axial 4407, coronal 4248, sagittal 4150).
# Set this to the three planes to reproduce the multi-plane idea; it costs
# about 3x the feature extraction.
PLANES = ["axial_fs"]
# PLANES = ["sagittal_fs", "coronal_fs", "axial_fs"]

EPOCHS, BATCH, LR, WD = 80, 16, 3e-4, 1e-4
use_amp = torch.cuda.is_available()                          # FIX 3

## Series table and plane keys

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))


def get_series_key(row):
    plane = str(row.get("Anatomical_Plane", "")).strip().lower()
    fluid = int(row.get("Fluid_Sensitive", 0) or 0)
    fat = int(row.get("Fat_Suppression", 0) or 0)
    return f"{plane}_fs" if (fluid == 1 and fat == 1) else f"{plane}_other"


train_series["series_key"] = train_series.apply(get_series_key, axis=1)

multi_plane_df = train_series[train_series["series_key"].isin(PLANES)].copy()
multi_plane_df = (multi_plane_df
                  .sort_values(["StudyInstanceUID", "series_key", "SeriesInstanceUID"])
                  .drop_duplicates(["StudyInstanceUID", "series_key"], keep="first"))

print("plane coverage over", len(train_df), "studies:")
for key in PLANES:
    n = multi_plane_df.loc[multi_plane_df["series_key"] == key,
                           "StudyInstanceUID"].nunique()
    print(f"  {key:14s} {n:5d}  ({n/len(train_df)*100:5.1f}%)")

## Weak labelsYour merge, unchanged: the weak columns are renamed so they cannot overwritea real label, the merge is validated one-to-one, and only gaps are filled.

In [ ]:
weak = pd.read_csv(WEAK_CSV)
assert weak["StudyInstanceUID"].nunique() == len(weak)
grid = {0.0, 0.25, 0.5, 0.75, 1.0}
off_grid = set(np.unique(weak[TARGETS].to_numpy())) - grid
assert not off_grid, f"labels off the rubric grid: {off_grid}"

all_studies = train_df.drop_duplicates("StudyInstanceUID").copy()
complete_mask = all_studies[TARGETS].notna().all(axis=1)
clean_df = all_studies.loc[complete_mask].reset_index(drop=True).copy()   # FIX 3
weak_df = weak[~weak["StudyInstanceUID"].isin(set(clean_df["StudyInstanceUID"]))] \
            .reset_index(drop=True).copy()                                # FIX 3

assert set(weak_df["StudyInstanceUID"]).isdisjoint(set(clean_df["StudyInstanceUID"]))
print("fully labelled studies:", len(clean_df))
print("weak-label studies:    ", len(weak_df))

weak_targets = weak_df[["StudyInstanceUID"] + TARGETS].rename(
    columns={t: f"{t}_weak" for t in TARGETS})

train_aug = train_df.merge(weak_targets, on="StudyInstanceUID",
                           how="left", validate="one_to_one")
for t in TARGETS:
    train_aug[t] = train_aug[t].fillna(train_aug[f"{t}_weak"])
    train_aug.drop(columns=[f"{t}_weak"], inplace=True)
print("rows with at least one usable label:",
      int(train_aug[TARGETS].notna().any(axis=1).sum()))

## FIX 1 and 2 — cache only what is used, then build `series_by_study`The original ran `cache_series_features` over 14,010 series: every study, everyplane. Only the studies that are trained or validated on are needed, which is`clean_df` plus `weak_df` = 1,778. With axial only that is 1,778 series ratherthan 14,010, roughly an eight-fold reduction, which is the difference betweenfinishing inside a session and not.`series_by_study` maps study -> plane -> cached `.npy` path. Your Dataset readsit; it was never built.

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load_dicom_series(series_path):
    dicoms = []
    for fname in os.listdir(series_path):
        fpath = os.path.join(series_path, fname)
        if not os.path.isfile(fpath):
            continue
        try:
            ds = pydicom.dcmread(fpath)
        except Exception:
            continue
        if hasattr(ds, "PixelData"):
            dicoms.append(ds)
    if not dicoms:
        raise RuntimeError(f"no DICOM images found: {series_path}")

    if hasattr(dicoms[0], "ImageOrientationPatient") and \
       hasattr(dicoms[0], "ImagePositionPatient"):
        iop = np.asarray(dicoms[0].ImageOrientationPatient, dtype=np.float32)
        normal = np.cross(iop[:3], iop[3:])
        order = np.argsort([
            float(np.dot(np.asarray(ds.ImagePositionPatient, dtype=np.float32), normal))
            for ds in dicoms])
        dicoms = [dicoms[i] for i in order]
    elif all(hasattr(ds, "InstanceNumber") for ds in dicoms):
        dicoms = sorted(dicoms, key=lambda ds: int(ds.InstanceNumber))

    volume = []
    for ds in dicoms:
        arr = ds.pixel_array.astype(np.float32)
        lo, hi = float(arr.min()), float(arr.max())
        volume.append((arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr))
    return volume


def build_backbone():
    bb = models.convnext_small(weights=models.ConvNeXt_Small_Weights.DEFAULT)
    bb.classifier = nn.Identity()
    bb.requires_grad_(False)
    return bb.eval().to(DEVICE)


@torch.no_grad()
def cache_series_features(df, backbone, slice_batch=32):
    done = skipped = failed = 0
    df = df.drop_duplicates("SeriesInstanceUID")
    for _, row in df.iterrows():
        out_path = os.path.join(FEATURE_CACHE, f"{row['SeriesInstanceUID']}.npy")
        if os.path.exists(out_path):
            skipped += 1
            continue
        try:
            volume = np.asarray(load_dicom_series(row["series_path"]))
            x = torch.stack([
                image_transform(Image.fromarray(
                    (np.clip(s, 0, 1) * 255).astype(np.uint8)).convert("RGB"))
                for s in volume]).to(DEVICE)
            feats = torch.cat([backbone(x[i:i + slice_batch]).cpu()
                               for i in range(0, len(x), slice_batch)])
            np.save(out_path, feats.reshape(len(x), -1).numpy().astype(np.float16))
            done += 1
        except Exception as exc:
            failed += 1
            if failed <= 5:
                print(f"  failed {row['SeriesInstanceUID']}: {exc}")
        if (done + skipped + failed) % 200 == 0:
            print(f"  {done + skipped + failed} | new={done} cached={skipped} "
                  f"failed={failed}", flush=True)
    print(f"finished: new={done}, already cached={skipped}, failed={failed}")


# series paths from the directory tree
series_paths = []
for study_uid in os.listdir(TRAIN_SERIES_ROOT):
    study_path = os.path.join(TRAIN_SERIES_ROOT, study_uid)
    if not os.path.isdir(study_path):
        continue
    for series_uid in os.listdir(study_path):
        p = os.path.join(study_path, series_uid)
        if os.path.isdir(p):
            series_paths.append({"StudyInstanceUID": study_uid,
                                 "SeriesInstanceUID": series_uid,
                                 "series_path": p})
series_paths_df = pd.DataFrame(series_paths)

multi_plane_df = multi_plane_df.drop(columns=["series_path"], errors="ignore").merge(
    series_paths_df, on=["StudyInstanceUID", "SeriesInstanceUID"],
    how="left", validate="many_to_one")

# FIX 1: only the studies actually used
needed_studies = set(clean_df["StudyInstanceUID"]) | set(weak_df["StudyInstanceUID"])
series_to_cache = multi_plane_df[
    multi_plane_df["StudyInstanceUID"].isin(needed_studies)
    & multi_plane_df["series_path"].notna()].copy()

print(f"studies needed: {len(needed_studies)}")
print(f"series to cache: {len(series_to_cache)}  "
      f"(unrestricted, all planes, would be {len(train_series[train_series['series_key'].str.endswith('_fs')])})")
print(series_to_cache["series_key"].value_counts().to_string())

backbone = build_backbone()
cache_series_features(series_to_cache, backbone)
del backbone
torch.cuda.empty_cache()

In [ ]:
# FIX 2: study -> plane -> cached feature path
series_by_study = {}
for _, r in series_to_cache.iterrows():
    path = os.path.join(FEATURE_CACHE, f"{r['SeriesInstanceUID']}.npy")
    if os.path.exists(path):
        series_by_study.setdefault(r["StudyInstanceUID"], {})[r["series_key"]] = path

print("studies with at least one cached plane:", len(series_by_study))
for key in PLANES:
    print(f"  {key:14s} {sum(1 for v in series_by_study.values() if key in v)}")

## Dataset and collate — yours, unchanged

In [ ]:
class KneeMultiPlaneFeatureDataset(Dataset):
    def __init__(self, dataframe, weak_dataframe=None, is_train=False, feature_dim=768):
        self.df = dataframe.reset_index(drop=True).copy()
        self.is_train = is_train
        self.feature_dim = feature_dim
        self.labels = (self.df[TARGETS].apply(pd.to_numeric, errors="coerce")
                       .to_numpy(dtype=np.float32))
        self.label_mask = np.isfinite(self.labels)
        self.labels = np.nan_to_num(self.labels, nan=0.0)
        self.weak_labels = np.zeros((len(self.df), len(TARGETS)), dtype=np.float32)
        self.weak_mask = np.zeros_like(self.weak_labels, dtype=bool)
        if weak_dataframe is not None and len(weak_dataframe) > 0:
            w = weak_dataframe.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")
            wn = w[TARGETS].apply(pd.to_numeric, errors="coerce")
            for i, sid in enumerate(self.df["StudyInstanceUID"]):
                if sid not in wn.index:
                    continue
                values = wn.loc[sid].to_numpy(dtype=np.float32)
                valid = np.isfinite(values)
                self.weak_labels[i, valid] = values[valid]
                self.weak_mask[i, valid] = True
        self.weak_mask &= ~self.label_mask

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sid = self.df.iloc[idx]["StudyInstanceUID"]
        available = series_by_study.get(sid, {})
        features, lengths, plane_present = {}, {}, {}
        for plane in PLANES:
            path = available.get(plane)
            if path is not None and os.path.isfile(path):
                arr = np.load(path).astype(np.float32)
                if arr.ndim != 2 or arr.shape[1] != self.feature_dim:
                    raise ValueError(f"unexpected shape {arr.shape} for {sid}/{plane}")
                if len(arr) > 0:
                    features[plane] = torch.from_numpy(arr)
                    lengths[plane] = len(arr)
                    plane_present[plane] = True
                    continue
            features[plane] = torch.zeros((1, self.feature_dim), dtype=torch.float32)
            lengths[plane] = 1
            plane_present[plane] = False
        return {"features": features, "lengths": lengths,
                "plane_present": plane_present,
                "labels": torch.from_numpy(self.labels[idx].copy()),
                "label_mask": torch.from_numpy(self.label_mask[idx].copy()),
                "weak_labels": torch.from_numpy(self.weak_labels[idx].copy()),
                "weak_mask": torch.from_numpy(self.weak_mask[idx].copy()),
                "study_id": sid}


def collate_multi_plane_features(batch):
    bs = len(batch)
    bf, bl, bp = {}, {}, {}
    for plane in PLANES:
        lengths = torch.tensor([item["lengths"][plane] if item["plane_present"][plane]
                                else 0 for item in batch], dtype=torch.long)
        max_len = max(int(lengths.max().item()), 1)
        padded = torch.zeros(bs, max_len, 768, dtype=torch.float32)
        present = torch.tensor([item["plane_present"][plane] for item in batch],
                               dtype=torch.bool)
        for i, item in enumerate(batch):
            if present[i]:
                seq = item["features"][plane]
                padded[i, :len(seq)] = seq
        bf[plane], bl[plane], bp[plane] = padded, lengths, present
    return {"features": bf, "lengths": bl, "plane_present": bp,
            "labels": torch.stack([x["labels"] for x in batch]),
            "label_mask": torch.stack([x["label_mask"] for x in batch]),
            "weak_labels": torch.stack([x["weak_labels"] for x in batch]),
            "weak_mask": torch.stack([x["weak_mask"] for x in batch]),
            "study_id": [x["study_id"] for x in batch]}

## Model — yours, unchanged

In [ ]:
class CachedFeatureMultiPlaneBiLSTM(nn.Module):
    def __init__(self, feature_dim=768, hidden_dim=128, num_layers=1,
                 num_targets=12, dropout=0.3, num_planes=3):
        super().__init__()
        self.num_planes = num_planes
        self.lstm = nn.LSTM(input_size=feature_dim, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.sequence_dim = 2 * hidden_dim
        self.norm = nn.LayerNorm(self.sequence_dim)
        self.pool_queries = nn.Parameter(torch.randn(num_planes, self.sequence_dim) * 0.02)
        self.missing_plane = nn.Parameter(torch.zeros(num_planes, self.sequence_dim))
        self.classifier = nn.Sequential(
            nn.Linear(num_planes * self.sequence_dim, 256), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(256, num_targets))

    def encode_plane(self, features, lengths, plane_present, plane_idx):
        B = features.shape[0]
        device = features.device
        pooled = self.missing_plane[plane_idx].unsqueeze(0).expand(B, -1).clone()
        valid_idx = torch.where((lengths > 0) & plane_present.bool())[0]
        if valid_idx.numel() == 0:
            return pooled
        valid_features = features[valid_idx]
        valid_lengths = lengths[valid_idx].long()
        packed = pack_padded_sequence(valid_features, valid_lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed)
        sequence, _ = pad_packed_sequence(packed_output, batch_first=True)
        sequence = self.norm(sequence)
        scores = torch.einsum("bsd,d->bs", sequence,
                              self.pool_queries[plane_idx]) / (self.sequence_dim ** 0.5)
        positions = torch.arange(sequence.shape[1], device=device).unsqueeze(0)
        mask = positions < valid_lengths.to(device).unsqueeze(1)
        scores = scores.masked_fill(~mask, float("-inf"))
        attention = torch.softmax(scores, dim=1)
        pooled[valid_idx] = (sequence * attention.unsqueeze(-1)).sum(dim=1)
        return pooled

    def forward(self, features, lengths, plane_present):
        pf = [self.encode_plane(features[p], lengths[p], plane_present[p], i)
              for i, p in enumerate(PLANES)]
        return self.classifier(torch.cat(pf, dim=1))


def prepare_multiplane_inputs(batch):
    features = {p: batch["features"][p].to(DEVICE, dtype=torch.float32,
                                           non_blocking=True) for p in PLANES}
    # FIX 4: lengths must reach encode_plane on DEVICE, because it does
    # `(lengths > 0) & plane_present.bool()` and plane_present is on DEVICE.
    # The model already calls .cpu() where pack_padded_sequence needs it.
    lengths = {p: batch["lengths"][p].to(DEVICE) for p in PLANES}
    plane_present = {p: batch["plane_present"][p].to(DEVICE, dtype=torch.bool,
                                                     non_blocking=True) for p in PLANES}
    return features, lengths, plane_present


def masked_bce_loss(logits, labels, label_mask, weak_labels, weak_mask, pos_weight=None):
    label_mask, weak_mask = label_mask.bool(), weak_mask.bool()
    targets = torch.where(label_mask, labels, weak_labels)
    target_mask = label_mask | weak_mask
    el = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight,
                                            reduction="none")
    return (el * target_mask.float()).sum() / target_mask.float().sum().clamp_min(1)

## FIX 5 — 5-fold CV over all 58 labelled studiesInstead of one 11-study validation set. Out-of-fold predictions are pooled andone macro AUC is computed over all 58, so every labelled study is used forevaluation exactly once and nothing is selected on it. The weak-label studiesare disjoint from all 58 and so are in training for every fold.

In [ ]:
def compute_pos_weight(frame, max_weight=20.0):
    ws = []
    for t in TARGETS:
        v = pd.to_numeric(frame[t], errors="coerce").to_numpy(np.float64)
        v = v[np.isfinite(v)]
        pos, neg = int((v == 1).sum()), int((v == 0).sum())
        ws.append(1.0 if (pos == 0 or neg == 0) else min(neg / pos, max_weight))
    return torch.tensor(ws, dtype=torch.float32)


def macro_auc(y_true, y_pred):
    aucs = []
    for i in range(len(TARGETS)):
        ok = np.isfinite(y_true[:, i]) & np.isfinite(y_pred[:, i])
        yt, yp = y_true[ok, i], y_pred[ok, i]
        aucs.append(roc_auc_score(yt, yp) if len(np.unique(yt)) == 2 else np.nan)
    finite = [a for a in aucs if np.isfinite(a)]
    return (float(np.mean(finite)) if finite else np.nan), aucs


def run_cv(seed, n_splits=5):
    Y = clean_df[TARGETS].to_numpy(np.float32)
    strata = pd.qcut(pd.Series(Y.sum(1)).rank(method="first"), q=n_splits, labels=False)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=20261020)
    oof = np.full_like(Y, np.nan, dtype=np.float64)

    for fold, (tr, ev) in enumerate(skf.split(np.zeros(len(Y)), strata)):
        train_frame = pd.concat([clean_df.iloc[tr], weak_df], ignore_index=True)
        val_frame = clean_df.iloc[ev].reset_index(drop=True)
        tr_ds = KneeMultiPlaneFeatureDataset(train_frame, weak_dataframe=weak_df,
                                             is_train=True)
        ev_ds = KneeMultiPlaneFeatureDataset(val_frame, weak_dataframe=None)
        tr_dl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, num_workers=2,
                           collate_fn=collate_multi_plane_features,
                           generator=torch.Generator().manual_seed(seed + 1000))
        ev_dl = DataLoader(ev_ds, batch_size=BATCH, shuffle=False, num_workers=2,
                           collate_fn=collate_multi_plane_features)

        torch.manual_seed(seed)
        model = CachedFeatureMultiPlaneBiLSTM(num_planes=len(PLANES)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        pw = compute_pos_weight(train_frame).to(DEVICE)

        for _ in range(EPOCHS):
            model.train()
            for batch in tr_dl:
                f, l, p = prepare_multiplane_inputs(batch)
                loss = masked_bce_loss(
                    model(f, l, p),
                    batch["labels"].to(DEVICE), batch["label_mask"].to(DEVICE),
                    batch["weak_labels"].to(DEVICE), batch["weak_mask"].to(DEVICE),
                    pos_weight=pw)
                opt.zero_grad(); loss.backward(); opt.step()

        model.eval()
        preds = []
        with torch.no_grad():
            for batch in ev_dl:
                f, l, p = prepare_multiplane_inputs(batch)
                preds.append(torch.sigmoid(model(f, l, p)).float().cpu().numpy())
        oof[ev] = np.concatenate(preds).astype(np.float64)
        print(f"  seed {seed} fold {fold} done", flush=True)

    return macro_auc(clean_df[TARGETS].to_numpy(np.float64), oof)


results = [run_cv(s) for s in (0, 1, 2)]
aucs = [r[0] for r in results]
print(f"\nPLANES={PLANES}")
print(f"5-fold CV over {len(clean_df)} labelled studies: "
      f"{np.mean(aucs):.4f} +- {np.std(aucs):.4f}   seeds {[round(a,4) for a in aucs]}")
print("\nmean-pool reference (56 studies): sagittal 0.7332 | axial 0.7514 | "
      "3-plane 0.7627")

per = np.nanmean([r[1] for r in results], axis=0)
print("\nper target:")
for i, t in enumerate(TARGETS):
    print(f"  {t:18s} {per[i]:.3f}")

## Reading the resultCompare against **0.7332 / 0.7514 / 0.7627**, not against the 0.6805 from theall-planes notebook or anything measured on 11 studies.Judge a difference against about **0.01**, not against the seed spread. Thatfigure is measured, not assumed: appending a constant, information-free columnto the mean-pooled features moved macro AUC by +0.0076 on sagittal and +0.0109on axial. A gap smaller than roughly 0.02 on this data is not evidence.I ran your model class verbatim through this protocol at one plane and at
three, and then tested the fusion question directly. The whole ladder:

    your BiLSTM, axial only                 0.7216 +- 0.0111
    your BiLSTM, 3 planes                   0.7458 +- 0.0088
    mean-pool head, axial alone             0.7514 +- 0.0105
    learned softmax plane weights           0.7557 +- 0.0036
    learned per-plane gate                  0.7573 +- 0.0026
    mean-pool head, 3 planes concatenated   0.7632 +- 0.0045

Against the best single plane (0.7514), the detection threshold is +0.02, and
nothing clears it. Plain concatenation is the best fusion at +0.012; the two
learned fusions are WORSE than concatenation (-0.006 and -0.007). So extra
machinery for combining planes does not reach anything that averaging and
concatenating misses.

Your architecture sits below the mean-pool head at both plane counts, by
0.030 at one plane and 0.017 at three. Its deficit narrows as planes are
added, which is why three planes look like a +0.024 gain inside it — that is
the architecture recovering from its own handicap, not new information being
unlocked. The giveaway is that the 3-plane run still sits below both plain
concatenation and single-plane axial.

So the recommendation is the simpler pipeline: mean-pool features, one good
plane, and the weak labels. That is also the conclusion the whole ladder
points at — on frozen features, adding INFORMATION helps and adding CAPACITY
does not. Weak labels +0.124 and a better plane +0.035 are information; six
architecture changes are capacity, and none of them paid.

The weak labels remain the firmly established part: +0.124 (0.609 -> 0.733),
eleven to fifteen times the noise floor, reproduced in three separate
implementations. That is worth keeping whatever the architecture does.